<a href="https://colab.research.google.com/github/EnzoAA004/PFI_MVPTest_Enzo_AImodule/blob/research%2Fpost-e50-axial-level-pairing/70_postE50_preprocessing_parity_automatic_disc_crop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# 70 · Post-E50 — P10.7 Preprocessing Parity & Automatic Disc Crop

Objetivo: cerrar el blocker que quedó después de 69: demostrar si el preprocessing real de producto puede reproducir el crop 2.5D usado por P10.7.

Este notebook es deliberadamente conservador:

- **NO entrena**.
- **NO abre** outer validation, internal test RSNA ni test oficial.
- **NO declara PASS** si no encuentra evidencia suficiente del preprocessing original.
- Reutiliza el checkpoint congelado P10.7.
- Toma 67B2 como upstream de nivel absoluto validado con abstención, pero **no confunde localización anatómica con paridad de crop**.
- Primero reconstruye el contrato de preprocessing desde artifacts/notebooks existentes.
- Luego deja preparado el generador de crops automáticos para comparar contra evidencia original.
- Si falta evidencia reproducible, cierra como `PREPROCESSING_PARITY_NOT_PROVEN` y no inventa parámetros.

## Índice

- 70-A — Scope + artifact identity
- 70-B — Inventario de evidencia original de preprocessing
- 70-C — Reconstrucción del contrato 2.5D
- 70-D — Automatic crop builder (fail-closed)
- 70-E — Parity decision + contract final

**Runtime recomendado:** CPU para A/B/C/E. GPU sólo si más adelante se agrega inferencia masiva; este notebook base puede correrse en CPU.


In [2]:

# ============================================================
# 70-A — SCOPE + ARTIFACT IDENTITY
# ============================================================

import hashlib, json, os, re, subprocess, sys, time
from datetime import datetime, timezone
from pathlib import Path
from typing import Mapping
import numpy as np
import pandas as pd

try:
    import google.colab
    IN_COLAB=True
except ImportError:
    IN_COLAB=False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

PFI_DRIVE_ROOT=Path(os.environ.get(
    'PFI_DRIVE_ROOT',
    '/content/drive/MyDrive/PFI_MVP' if IN_COLAB else './PFI_MVP'
))

RESULTS_DIR=PFI_DRIVE_ROOT/'results'/'post_e50'/'70'
METRICS_DIR=PFI_DRIVE_ROOT/'metrics'/'post_e50'/'70'
RESULTS_DIR.mkdir(parents=True,exist_ok=True)
METRICS_DIR.mkdir(parents=True,exist_ok=True)

PATHOLOGY_CHECKPOINT_PATH=(
    PFI_DRIVE_ROOT/'models'/'P10_7_spider_degenerative'/
    'final_internal_test_evaluation'/
    'frozen_p10_7_spider_degenerative_multitask.pt'
)

EXPECTED_PATHOLOGY_SHA256='16eccff327e6794b127fe372ecd03ea619a0f69d939b84ae1aa2e904191c6293'
EXPECTED_SCHEMA_VERSION='pfi.p10-7-research-export.v1'

TRAINING_PERFORMED=False
RSNA_OUTER_VALIDATION_ACCESSED=False
RSNA_INTERNAL_TEST_ACCESSED=False
OFFICIAL_TEST_ACCESSED=False

assert not TRAINING_PERFORMED
assert not RSNA_OUTER_VALIDATION_ACCESSED
assert not RSNA_INTERNAL_TEST_ACCESSED
assert not OFFICIAL_TEST_ACCESSED

def sha256_file(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''):
            h.update(chunk)
    return h.hexdigest()

assert PATHOLOGY_CHECKPOINT_PATH.is_file(), PATHOLOGY_CHECKPOINT_PATH
PATHOLOGY_SHA256=sha256_file(PATHOLOGY_CHECKPOINT_PATH)
assert PATHOLOGY_SHA256==EXPECTED_PATHOLOGY_SHA256

print('='*72)
print('70-A — SCOPE + ARTIFACT IDENTITY')
print('='*72)
print('Checkpoint:',PATHOLOGY_CHECKPOINT_PATH)
print('SHA256:',PATHOLOGY_SHA256)
print('Scope: NO TRAINING / NO RSNA VALIDATION / NO INTERNAL TEST / NO OFFICIAL TEST')
print('67B2 upstream localization: VALIDATED_WITH_ABSTENTION')
print('70-A: PASS')


Mounted at /content/drive
70-A — SCOPE + ARTIFACT IDENTITY
Checkpoint: /content/drive/MyDrive/PFI_MVP/models/P10_7_spider_degenerative/final_internal_test_evaluation/frozen_p10_7_spider_degenerative_multitask.pt
SHA256: 16eccff327e6794b127fe372ecd03ea619a0f69d939b84ae1aa2e904191c6293
Scope: NO TRAINING / NO RSNA VALIDATION / NO INTERNAL TEST / NO OFFICIAL TEST
67B2 upstream localization: VALIDATED_WITH_ABSTENTION
70-A: PASS


In [3]:

# ============================================================
# 70-B — INVENTORY ORIGINAL PREPROCESSING EVIDENCE
# ============================================================

# Buscamos SOLO dentro de PFI_MVP y con nombres/patrones acotados.
# No se abre dataset de evaluación. Sólo código, manifests y artifacts técnicos.

SEARCH_ROOTS=[
    PFI_DRIVE_ROOT/'models'/'P10_7_spider_degenerative',
    PFI_DRIVE_ROOT/'notebooks',
    PFI_DRIVE_ROOT/'research',
    PFI_DRIVE_ROOT/'artifacts',
    PFI_DRIVE_ROOT/'results',
]

NAME_HINTS=[
    'p10_7','p10-7','spider','degenerative','multitask',
    'disc','ivd','preprocess','crop','2p5d','2.5d'
]

evidence_files=[]

for root in SEARCH_ROOTS:
    if not root.exists():
        continue
    for p in root.rglob('*'):
        if not p.is_file():
            continue
        low=p.name.lower()
        if any(h in low for h in NAME_HINTS):
            if p.suffix.lower() in {'.ipynb','.py','.json','.md','.txt','.csv','.yaml','.yml'}:
                evidence_files.append(p)

evidence_files=sorted(set(evidence_files))

rows=[]
for p in evidence_files:
    try:
        rel=str(p.relative_to(PFI_DRIVE_ROOT))
    except Exception:
        rel=str(p)
    rows.append({
        'path':rel,
        'suffix':p.suffix.lower(),
        'size_bytes':p.stat().st_size,
    })

evidence_df=pd.DataFrame(rows)
EVIDENCE_CSV=RESULTS_DIR/'70_B_preprocessing_evidence_inventory.csv'
evidence_df.to_csv(EVIDENCE_CSV,index=False)

print('='*72)
print('70-B — PREPROCESSING EVIDENCE INVENTORY')
print('='*72)
print('Evidence files found:',len(evidence_df))

if not evidence_df.empty:
    print(evidence_df[['path','suffix','size_bytes']].to_string(index=False,max_rows=80))

print('Inventory SHA256:',sha256_file(EVIDENCE_CSV))


70-B — PREPROCESSING EVIDENCE INVENTORY
Evidence files found: 29
                                                                                                                                 path suffix  size_bytes
                     models/P10_7_spider_degenerative/final_internal_test_evaluation/disc_degenerative_findings_schema_candidate.json  .json         722
                                                                           results/E3_E4_exploracion/E3_E4_spider_dataset_summary.csv   .csv      123072
                                                                        results/E4_preprocesamiento/E4_spider_consistency_summary.csv   .csv      424635
                                                                                      results/E4_preprocesamiento/E4_spider_pairs.csv   .csv       71308
                                                    results/E8_alkafri_official_pairing/E8_FIXED_gt_sources_by_case_modality_disc.csv   .csv      831267
                 

In [4]:

# ============================================================
# 70-C — EXTRACT PREPROCESSING CONTRACT FROM EXISTING SOURCES
# ============================================================

# Known invariants already established by P10.7 runtime/docs.
KNOWN_INVARIANTS={
    'image_size':224,
    'channels_per_modality':3,
    'series_roles':['sagittal_t1','sagittal_t2'],
    'spine_levels':['L1-L2','L2-L3','L3-L4','L4-L5','L5-S1'],
    'ivd_label_by_level':{
        'L1-L2':1,
        'L2-L3':2,
        'L3-L4':3,
        'L4-L5':4,
        'L5-S1':5,
    },
    'raw_disc_label_formula':'200 + ivd_label',
    'ivd_index_formula':'ivd_label - 1',
    'checkpoint_sha256':PATHOLOGY_SHA256,
}

PATTERNS={
    'raw_disc_label':re.compile(r'raw_disc_label|200\s*\+\s*ivd_label',re.I),
    'ivd_index':re.compile(r'ivd_index|ivd_label\s*-\s*1',re.I),
    'image_size_224':re.compile(r'224',re.I),
    'crop':re.compile(r'\bcrop\b|bounding|bbox|roi',re.I),
    '2p5d':re.compile(r'2\.5d|2p5d|three.slice|3.slice|neighbor',re.I),
    'percentile_norm':re.compile(r'percentile|p1|p99|clip|normaliz',re.I),
    'resize':re.compile(r'resize|interpol',re.I),
    't1':re.compile(r'sagittal[_\s-]*t1',re.I),
    't2':re.compile(r'sagittal[_\s-]*t2|t2/stir',re.I),
}

def read_textish(path):
    suf=path.suffix.lower()
    try:
        if suf=='.ipynb':
            data=json.loads(path.read_text(encoding='utf-8'))
            chunks=[]
            for cell in data.get('cells',[]):
                src=cell.get('source',[])
                if isinstance(src,list):
                    chunks.extend(src)
                elif isinstance(src,str):
                    chunks.append(src)
            return '\n'.join(chunks)
        return path.read_text(encoding='utf-8',errors='ignore')
    except Exception:
        return ''

matches=[]
for p in evidence_files:
    txt=read_textish(p)
    if not txt:
        continue
    hit={k:bool(rx.search(txt)) for k,rx in PATTERNS.items()}
    score=sum(hit.values())
    if score:
        matches.append({
            'path':str(p.relative_to(PFI_DRIVE_ROOT)) if str(p).startswith(str(PFI_DRIVE_ROOT)) else str(p),
            'score':score,
            **hit
        })

match_df=pd.DataFrame(matches)
if not match_df.empty:
    match_df=match_df.sort_values(['score','path'],ascending=[False,True])

MATCH_CSV=RESULTS_DIR/'70_C_preprocessing_source_matches.csv'
match_df.to_csv(MATCH_CSV,index=False)

print('='*72)
print('70-C — SOURCE MATCH AUDIT')
print('='*72)

if match_df.empty:
    print('No readable preprocessing evidence matched.')
else:
    print(match_df.head(50).to_string(index=False))

# Gate conservador:
# los invariantes documentados NO alcanzan para probar crop parity.
exact_crop_source_found=False
normalization_source_found=False
slice_stack_source_found=False

if not match_df.empty:
    # requiere que una MISMA fuente tenga crop + 2.5D + resize + al menos una referencia de normalización
    exact_candidates=match_df[
        (match_df['crop']==True) &
        (match_df['2p5d']==True) &
        (match_df['resize']==True)
    ]
    exact_crop_source_found=len(exact_candidates)>0

    normalization_source_found=bool(match_df['percentile_norm'].any())
    slice_stack_source_found=bool(match_df['2p5d'].any())

contract_reconstruction={
    'stage':'70-C',
    'known_invariants':KNOWN_INVARIANTS,
    'evidence_files_count':int(len(evidence_df)),
    'matched_sources_count':int(len(match_df)),
    'exact_crop_source_candidate_found':bool(exact_crop_source_found),
    'normalization_source_candidate_found':bool(normalization_source_found),
    'slice_stack_source_candidate_found':bool(slice_stack_source_found),
    'parity_proven':False,
    'reason':'Source discovery only; exact numerical equivalence has not yet been demonstrated.',
}

RECON_PATH=METRICS_DIR/'70_C_contract_reconstruction.json'
RECON_PATH.write_text(json.dumps(contract_reconstruction,indent=2))

print()
print('Exact crop source candidate:',exact_crop_source_found)
print('Normalization source candidate:',normalization_source_found)
print('2.5D stack source candidate:',slice_stack_source_found)
print('PARITY PROVEN: False')
print('Source matches SHA256:',sha256_file(MATCH_CSV))
print('Reconstruction SHA256:',sha256_file(RECON_PATH))


70-C — SOURCE MATCH AUDIT
                                                                                                   path  score  raw_disc_label  ivd_index  image_size_224  crop  2p5d  percentile_norm  resize    t1    t2
                                           results/P10_7_spider_degenerative/disc_level_manifest_v1.csv      4            True      False            True  True False             True   False False False
                                          results/E4_preprocesamiento/E4_spider_consistency_summary.csv      2           False      False            True False False             True   False False False
                                          results/GCS_spider_final_training_v1/spider_cache_receipt.csv      2           False      False            True False False             True   False False False
                                    results/GCS_spider_final_training_v1/spider_class_distribution.json      2           False      False            True False Fa

In [5]:

# ============================================================
# 70-D — AUTOMATIC CROP BUILDER CONTRACT (FAIL-CLOSED)
# ============================================================

# Esta celda NO inventa los parámetros desconocidos del training preprocessing.
# Construye solamente la interfaz y valida los invariantes seguros.

SPINE_LEVELS=['L1-L2','L2-L3','L3-L4','L4-L5','L5-S1']

def level_mapping(level):
    if level not in SPINE_LEVELS:
        raise ValueError('unsupported_lumbar_level')
    ivd_label=SPINE_LEVELS.index(level)+1
    return {
        'level':level,
        'ivd_label':ivd_label,
        'ivd_index':ivd_label-1,
        'raw_disc_label':200+ivd_label,
    }

for lvl in SPINE_LEVELS:
    m=level_mapping(lvl)
    assert m['raw_disc_label']==200+m['ivd_label']
    assert m['ivd_index']==m['ivd_label']-1

# Campos que DEBEN quedar definidos antes de habilitar crop automático real.
REQUIRED_EXACT_PREPROCESSING_FIELDS=[
    'disc_roi_definition',
    'roi_margin_rule',
    'slice_selection_rule_3_channels',
    'series_alignment_rule_t1_t2',
    'pixel_intensity_normalization',
    'resize_interpolation',
    'mask_to_roi_coordinate_rule',
]

# Por ahora sólo marcamos lo realmente corroborado.
preprocessing_spec={
    'image_size':[224,224],
    'channels_per_modality':3,
    'level_mapping':{lvl:level_mapping(lvl) for lvl in SPINE_LEVELS},
    'disc_roi_definition':None,
    'roi_margin_rule':None,
    'slice_selection_rule_3_channels':None,
    'series_alignment_rule_t1_t2':None,
    'pixel_intensity_normalization':None,
    'resize_interpolation':None,
    'mask_to_roi_coordinate_rule':None,
}

missing_exact=[
    k for k in REQUIRED_EXACT_PREPROCESSING_FIELDS
    if preprocessing_spec.get(k) is None
]

AUTO_CROP_ENABLED=len(missing_exact)==0

assert AUTO_CROP_ENABLED is False, (
    'Unexpected: automatic crop should remain disabled until exact preprocessing is filled from evidence.'
)

SPEC_PATH=METRICS_DIR/'70_D_preprocessing_spec_fail_closed.json'
SPEC_PATH.write_text(json.dumps({
    'stage':'70-D',
    'spec':preprocessing_spec,
    'missing_exact_fields':missing_exact,
    'automatic_crop_enabled':AUTO_CROP_ENABLED,
    'human_review_required':True,
},indent=2))

print('='*72)
print('70-D — AUTOMATIC CROP BUILDER CONTRACT')
print('='*72)
print('Safe level mapping: PASS')
print('Missing exact preprocessing fields:')
for k in missing_exact:
    print(' -',k)
print('AUTOMATIC CROP ENABLED:',AUTO_CROP_ENABLED)
print('Spec SHA256:',sha256_file(SPEC_PATH))


70-D — AUTOMATIC CROP BUILDER CONTRACT
Safe level mapping: PASS
Missing exact preprocessing fields:
 - disc_roi_definition
 - roi_margin_rule
 - slice_selection_rule_3_channels
 - series_alignment_rule_t1_t2
 - pixel_intensity_normalization
 - resize_interpolation
 - mask_to_roi_coordinate_rule
AUTOMATIC CROP ENABLED: False
Spec SHA256: e742812efe5c4891812d541fa1268e3764c1e7f024884b17e775dfea5dbc8a1f


In [6]:

# ============================================================
# 70-E — PARITY DECISION + FINAL CONTRACT
# ============================================================

PARITY_PASS=False

# Reglas:
# - descubrir código parecido NO es prueba;
# - 67B2 validado NO prueba crop parity;
# - un forward sintético P10.7 NO prueba preprocessing parity;
# - sólo se habilita cuando tengamos los parámetros exactos y una comparación reproducible.

decision={
    'stage':'70-E',
    'status':'PREPROCESSING_PARITY_NOT_PROVEN',
    'checkpoint_sha256':PATHOLOGY_SHA256,
    'upstream_localization':{
        'stage':'67B2',
        'status':'VALIDATED_WITH_ABSTENTION',
    },
    'source_inventory_complete_for_current_drive':True,
    'exact_crop_source_candidate_found':bool(exact_crop_source_found),
    'normalization_source_candidate_found':bool(normalization_source_found),
    'slice_stack_source_candidate_found':bool(slice_stack_source_found),
    'missing_exact_fields':missing_exact,
    'preprocessing_parity_validated':PARITY_PASS,
    'automatic_product_inference_ready':False,
    'training_required':False,
    'gpu_required_for_current_stage':False,
    'next_action':(
        'Inspect the highest-scoring original preprocessing source(s), '
        'fill exact crop/normalization/stack/alignment rules, then run a '
        'numerical parity smoke on development-only examples before enabling runtime.'
    ),
    'governance':{
        'training_performed':False,
        'rsna_outer_validation_accessed':False,
        'rsna_internal_test_accessed':False,
        'official_test_accessed':False,
    }
}

DECISION_PATH=METRICS_DIR/'70_E_preprocessing_parity_decision.json'
DECISION_PATH.write_text(json.dumps(decision,indent=2))

summary={
    'stage':'70',
    'status':decision['status'],
    'checkpoint_sha256':PATHOLOGY_SHA256,
    'preprocessing_parity_validated':False,
    'automatic_product_inference_ready':False,
    'training_required':False,
    'next_step':'70F_exact_source_reconstruction_then_parity_smoke',
    'inventory_sha256':sha256_file(EVIDENCE_CSV),
    'source_matches_sha256':sha256_file(MATCH_CSV),
    'reconstruction_sha256':sha256_file(RECON_PATH),
    'spec_sha256':sha256_file(SPEC_PATH),
    'decision_sha256':sha256_file(DECISION_PATH),
}

SUMMARY_PATH=METRICS_DIR/'70_E_final_summary.json'
SUMMARY_PATH.write_text(json.dumps(summary,indent=2))

print('='*72)
print('70-E — PREPROCESSING PARITY DECISION')
print('='*72)
print('Status:',decision['status'])
print('PREPROCESSING PARITY VALIDATED:',False)
print('AUTOMATIC PRODUCT INFERENCE READY:',False)
print('TRAINING REQUIRED:',False)
print('GPU REQUIRED NOW:',False)
print('Next step:',summary['next_step'])
print('Decision SHA256:',sha256_file(DECISION_PATH))
print('Final summary SHA256:',sha256_file(SUMMARY_PATH))


70-E — PREPROCESSING PARITY DECISION
Status: PREPROCESSING_PARITY_NOT_PROVEN
PREPROCESSING PARITY VALIDATED: False
AUTOMATIC PRODUCT INFERENCE READY: False
TRAINING REQUIRED: False
GPU REQUIRED NOW: False
Next step: 70F_exact_source_reconstruction_then_parity_smoke
Decision SHA256: 2e6881d37aa87f4ae94fef592838409747eb2df469eae060e7a645d1cd48f70e
Final summary SHA256: c2cea087879154380c278da77b4d9c25661cc1e22c000d90806c998b5f599c17
